# 08 - Premiers modèles de prédiction du prix au m²

**Objectif** : construire une baseline puis des modèles prédisant `prix_m2`, et évaluer honnêtement leur performance.

**Entrée** : `dvf_77_2025_enrichi.parquet` (13 712 ventes, sortie du notebook 06).
**Sortie** : aucun fichier (notebook d'expérimentation).

**Plan**
1. Préparation : cible, variables, découpage train/test, baseline
2. Régression linéaire et forêt aléatoire
3. Réglage de `min_samples_leaf` par validation croisée
4. Importance des variables (MDI)
5. Importance par permutation et position géographique
6. Validation par commune (communes jamais vues)
7. Évaluation finale sur le jeu de test

## 1. Préparation et baseline

- **Cible** : `prix_m2`.
- **Variables exclues** : `valeur_fonciere` (fuite de données : `prix_m2` en est le quotient par la surface) et `id_mutation` (identifiant unique, sans information généralisable).
- **Ventes sans GPS** : les 6 ventes sans distance sont écartées pour ce premier essai (13 706 ventes restent).
- **Découpage** aléatoire 80 % / 20 % (`random_state=42` pour la reproductibilité ; le fichier n'est pas mélangé, il est groupé par commune).
- **Baseline** : prédire la médiane du train pour toutes les ventes du test. Elle fixe le seuil à battre.

In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupKFold

df = pd.read_parquet("../data/processed/dvf_77_2025_enrichi.parquet")

# 1. Cible et variables explicatives (valeur_fonciere et id_mutation exclues)
cible = "prix_m2"
variables = ["type_local", "surface_reelle_bati", "nombre_pieces_principales",
             "nb_dependances", "surface_terrain_total", "distance_paris_km"]

# 2. Les 6 ventes sans GPS n'ont pas de distance : on les met de côté pour ce premier essai
df_modele = df.dropna(subset=["distance_paris_km"])

X = df_modele[variables]
y = df_modele[cible]

# 3. Découpage aléatoire 80 % / 20 %
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(X_train.shape, X_test.shape)

# 4. Baseline : prédire la médiane de l'entraînement pour toutes les ventes du test
mediane_train = y_train.median()
pred_baseline = np.full(len(y_test), mediane_train)
print("MAE baseline :", round(mean_absolute_error(y_test, pred_baseline)))

(10964, 6) (2742, 6)
MAE baseline : 887


**Lecture** : 10 964 ventes en entraînement, 2 742 en test. La baseline a une MAE de 887 €/m² (≈ 29 % d'un prix médian de 3 070 €/m²). Le choix de la graine fait varier cette valeur d'environ 40 €/m² (de 849 à 887 selon la graine testée) : ne pas choisir la graine d'après le score.

## 2. Régression linéaire et forêt aléatoire

`type_local` est encodé en 0/1 (Maison = 1). La règle est fixe, donc sans fuite de données depuis le test.

In [21]:
# 1. Encodage du type de bien : Maison = 1, Appartement = 0
X_train = X_train.copy()
X_test = X_test.copy()
X_train["type_local"] = (X_train["type_local"] == "Maison").astype(int)
X_test["type_local"] = (X_test["type_local"] == "Maison").astype(int)

# 2. Régression linéaire
modele_lin = LinearRegression()
modele_lin.fit(X_train, y_train)
pred_lin = modele_lin.predict(X_test)
print("MAE régression linéaire :", round(mean_absolute_error(y_test, pred_lin)))

# 3. Forêt aléatoire
modele_rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
modele_rf.fit(X_train, y_train)
pred_rf = modele_rf.predict(X_test)
print("MAE forêt aléatoire :", round(mean_absolute_error(y_test, pred_rf)))

MAE régression linéaire : 702
MAE forêt aléatoire : 617


**Lecture** : les deux modèles battent la baseline (887 €/m²) : -21 % pour la régression linéaire (702) et -30 % pour la forêt (617), qui capte des relations non linéaires et des interactions. Lors d'un essai, la forêt affichait une MAE de 227 €/m² sur le train contre 617 sur le test : un écart net, signe de sur-apprentissage.

## 3. Réglage de `min_samples_leaf` par validation croisée

On essaie plusieurs valeurs sur le **train uniquement** (5 parties) : le test reste intact jusqu'à l'évaluation finale.

In [22]:
for leaf in [1, 5, 10, 20]:
    rf = RandomForestRegressor(
        n_estimators=200, min_samples_leaf=leaf, random_state=42, n_jobs=-1
    )
    scores = cross_val_score(
        rf, X_train, y_train, cv=5, scoring="neg_mean_absolute_error"
    )
    print("min_samples_leaf =", leaf, "-> MAE moyenne :", round(-scores.mean()))

min_samples_leaf = 1 -> MAE moyenne : 604
min_samples_leaf = 5 -> MAE moyenne : 598
min_samples_leaf = 10 -> MAE moyenne : 602
min_samples_leaf = 20 -> MAE moyenne : 608


**Lecture** : 604 (1), 598 (5), 602 (10), 608 (20) €/m². Le réglage a très peu d'effet (≈ 1 %) : la forêt fait déjà la moyenne de 200 arbres, ce qui limite le sur-apprentissage. On garde `min_samples_leaf=5`. Le plafond vient surtout de l'information disponible, pas du réglage.

## 4. Importance des variables (MDI) et dispersion de la validation croisée

L'importance MDI mesure la réduction de variance des prix apportée par les questions de chaque variable, cumulée sur tous les arbres. Elle favorise les variables continues et partage l'importance entre variables corrélées.

In [23]:
# 1. Les écarts de la validation croisée sont-ils significatifs ?
rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1)
scores = -cross_val_score(rf, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
print("MAE par partie :", scores.round())
print("Moyenne :", round(scores.mean()), "| Écart-type :", round(scores.std()))

# 2. Quelles variables le modèle utilise-t-il le plus ?
rf.fit(X_train, y_train)
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances.round(3))

MAE par partie : [598. 600. 596. 591. 604.]
Moyenne : 598 | Écart-type : 4
distance_paris_km            0.545
surface_reelle_bati          0.252
surface_terrain_total        0.133
nb_dependances               0.028
nombre_pieces_principales    0.024
type_local                   0.019
dtype: float64


**Lecture** : l'écart-type entre les 5 parties est faible (4 €/m² autour de 598). `distance_paris_km` domine (0,545), devant la surface (0,252) et le terrain (0,133), cohérent avec le gradient du notebook 06.

## 5. Importance par permutation et position géographique

**Permutation** : on mélange une colonne du test et on mesure la hausse de la MAE ; calcul sur le test, en €/m², sans favoriser les variables à beaucoup de valeurs.

**Position géographique** : on ajoute `latitude` et `longitude` (la distance à Paris est un cercle autour de Paris ; les coordonnées donnent aussi la direction).

In [24]:
# 1. Importance par permutation (sur le test)

resultat = permutation_importance(
    rf, X_test, y_test, n_repeats=10, random_state=42,
    scoring="neg_mean_absolute_error", n_jobs=-1
)
imp_perm = pd.Series(resultat.importances_mean, index=X_test.columns).sort_values(ascending=False)
print(imp_perm.round(1))

# 2. Ajouter la position géographique (latitude, longitude)
variables_geo = variables + ["latitude", "longitude"]
X_geo = df_modele[variables_geo].copy()
X_geo["type_local"] = (X_geo["type_local"] == "Maison").astype(int)
Xg_train, Xg_test, yg_train, yg_test = train_test_split(
    X_geo, y, test_size=0.2, random_state=42
)

rf_geo = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1)
scores_geo = -cross_val_score(rf_geo, Xg_train, yg_train, cv=5, scoring="neg_mean_absolute_error")
print("MAE avec latitude/longitude :", round(scores_geo.mean()), "| Écart-type :", round(scores_geo.std()))

distance_paris_km            364.9
surface_reelle_bati          329.3
surface_terrain_total         79.2
type_local                    55.8
nb_dependances                16.4
nombre_pieces_principales      8.7
dtype: float64
MAE avec latitude/longitude : 513 | Écart-type : 10


**Lecture** : par permutation, la distance (365 €/m²) et la surface (329) pèsent presque autant ; `type_local` remonte (56) car le MDI le sous-estimait ; les pièces comptent peu (9), la surface les remplace. Avec latitude et longitude, la MAE en validation croisée passe de 598 à 513 €/m² (-14 %), bien au-delà de la dispersion (écart-type 10). Hypothèse (non vérifiée) : le modèle repère des zones (Fontainebleau, Marne-la-Vallée) que la distance seule ne distingue pas.

## 6. Validation par commune

Avec un découpage aléatoire, des ventes voisines (même immeuble, même programme neuf) se retrouvent dans le train et la validation : le modèle peut reconnaître le lieu (fuite spatiale). `GroupKFold` met toutes les ventes d'une commune dans la même partie ; `groups` fournit l'étiquette de chaque vente (son code commune) mais n'est jamais une variable du modèle.

In [25]:
groupes = df_modele.loc[X_train.index, "code_commune"]

def mae_par_commune(modele, X):
    scores = -cross_val_score(
        modele, X, y_train, cv=GroupKFold(n_splits=5), groups=groupes,
        scoring="neg_mean_absolute_error"
    )
    return round(scores.mean())

rf_base = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1)
print("Sans géo, par commune :", mae_par_commune(rf_base, X_train))
print("Avec géo, par commune :", mae_par_commune(rf_geo, Xg_train))

Sans géo, par commune : 688
Avec géo, par commune : 590


**Lecture** : sur des communes jamais vues, les deux modèles se dégradent de façon comparable (+90 €/m² sans géo : 598 vers 688 ; +77 avec géo : 513 vers 590). Le gain de la géo résiste (-98 €/m², -14 %) : il ne vient pas d'une fuite spatiale. `distance_paris_km` étant une fonction des coordonnées, le modèle sans géo connaissait déjà une partie de la localisation. Limite : les communes voisines restent dans le train (autocorrélation spatiale).

## 7. Évaluation finale sur le jeu de test

Modèle retenu : forêt aléatoire avec latitude et longitude, `min_samples_leaf=5`. Le test est utilisé **une seule fois** ; on ne retouche plus les réglages après ce résultat.

In [26]:
rf_final = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_final.fit(Xg_train, yg_train)
pred_final = rf_final.predict(Xg_test)
print("MAE test final :", round(mean_absolute_error(yg_test, pred_final)))

erreurs = pd.DataFrame({
    "type": df_modele.loc[Xg_test.index, "type_local"],
    "tranche_distance": pd.cut(df_modele.loc[Xg_test.index, "distance_paris_km"], [0, 20, 30, 40, 50, 60, 100]),
    "erreur_abs": (yg_test - pred_final).abs(),
})
print(erreurs.groupby("type")["erreur_abs"].agg(["mean", "count"]).round())
print(erreurs.groupby("tranche_distance", observed=True)["erreur_abs"].agg(["mean", "count"]).round())

MAE test final : 518
              mean  count
type                     
Appartement  429.0   1196
Maison       588.0   1546
                   mean  count
tranche_distance              
(0, 20]           527.0    176
(20, 30]          502.0    809
(30, 40]          511.0    574
(40, 50]          492.0    549
(50, 60]          654.0    270
(60, 100]         503.0    364


**Lecture** : la MAE test (518 €/m²) est cohérente avec la validation croisée (513). L'erreur est plus forte pour les maisons (588) que pour les appartements (429). Elle est stable selon la distance (≈ 490 à 530 €/m²) sauf entre 50 et 60 km (654, 270 ventes). Cette tranche cumule la dispersion de prix la plus forte (écart-type ≈ 1 390 €/m²) et un voisinage très hétérogène : Fontainebleau (MAE ≈ 900, prix moyen ≈ 4 480 €/m²) à côté de communes à 2 200-2 500 €/m² ; hors Fontainebleau et Bois-le-Roi, la MAE y reste de 576 €/m². Hypothèse (non vérifiée) : une prime propre à la zone de la forêt de Fontainebleau que les variables disponibles ne décrivent pas.

## Conclusion

- **Baseline** : 887 €/m² de MAE en prédisant la médiane.
- **Modèle retenu** : forêt aléatoire (200 arbres, `min_samples_leaf=5`) avec surface, terrain, dépendances, pièces, type de bien, distance à Paris, latitude et longitude. MAE de 518 €/m² sur le jeu de test, soit environ -42 % par rapport à la baseline. Sur des communes jamais vues, la validation croisée donne environ 590 €/m² : c'est le chiffre à annoncer pour un usage sur de nouvelles communes.
- **Ce qui compte** : la position (distance puis coordonnées) et la surface. Le type de bien et le nombre de pièces apportent peu une fois le reste connu.
- **Limites** : erreur plus forte pour les maisons et pour la zone de Fontainebleau ; une seule année ; pas d'information sur l'âge, l'état du bien ni la desserte ; les 574 maisons avec terrain à 0 (ambigu) et les 6 ventes sans GPS ne sont pas traitées finement ; validation par commune qui laisse les communes voisines dans le train.
- **Pistes** : encodage de la commune, variables externes (transports, zones commerciales), modèles de boosting, ré-entraînement sur plusieurs années.